<a href="https://colab.research.google.com/github/office268/ipynb/blob/main/ai_agent_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# סוכן AI עם LangGraph ו-Tavily

## מה נלמד?

במחברת זו נבנה **סוכן AI פשוט** שיודע לחפש מידע באינטרנט ולענות על שאלות.

### הכלים שנשתמש בהם:

| כלי | תפקיד |
|-----|--------|
| **LangGraph** | מסגרת לבניית סוכנים כ-graph של מצבים ופעולות |
| **Claude (Anthropic)** | מודל השפה שמחליט מתי לחפש ומה לענות |
| **Tavily** | מנוע חיפוש AI לחיפוש מידע עדכני מהאינטרנט |

### איך זה עובד?

```
שאלה → LLM → האם צריך לחפש? → כן → Tavily Search → LLM → תשובה
                                ↓ לא
                              תשובה ישירה
```

הסוכן רץ בלולאה עד שהמודל מחליט שיש מספיק מידע כדי לענות.

In [ ]:
# התקנת ספריות נדרשות
!pip install -q langgraph langchain-anthropic langchain-community tavily-python

In [ ]:
import os

# הגדרת מפתחות API מתוך Colab Secrets
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["TAVILY_API_KEY"]    = userdata.get("TAVILY_API_KEY")
except Exception:
    # הרצה מחוץ ל-Colab — הגדר ידנית:
    # os.environ["ANTHROPIC_API_KEY"] = "..."
    # os.environ["TAVILY_API_KEY"]    = "..."
    pass

print("ANTHROPIC_API_KEY:", "✓" if os.environ.get("ANTHROPIC_API_KEY") else "✗ חסר")
print("TAVILY_API_KEY:   ", "✓" if os.environ.get("TAVILY_API_KEY")    else "✗ חסר")

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_community.tools.tavily_search import TavilySearchResults

# כלי החיפוש — מחזיר עד 3 תוצאות
search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]

# המודל — Claude עם גישה לכלים
llm = ChatAnthropic(model="claude-sonnet-4-6").bind_tools(tools)

print("כלי חיפוש:", search_tool.name)
print("מודל:     ", "claude-sonnet-4-6")

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# --- Nodes ---

def llm_node(state: MessagesState):
    """המודל מקבל את ההיסטוריה ומחליט: לחפש או לענות."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)  # מריץ את כלי החיפוש אוטומטית

# --- Conditional edge ---

def should_continue(state: MessagesState):
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

# --- Graph ---

graph = StateGraph(MessagesState)
graph.add_node("llm",   llm_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "llm")
graph.add_conditional_edges("llm", should_continue)
graph.add_edge("tools", "llm")

agent = graph.compile()
print("הגרף נבנה בהצלחה ✓")

# הדמיית הגרף (אופציונלי — דורש Graphviz)
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print("START → llm → (tool_calls?) → tools → llm → ... → END")

In [ ]:
from langchain_core.messages import HumanMessage

question = "מה הם החידושים הכי מרגשים בתחום ה-AI בשנת 2025?"
print(f"שאלה: {question}\n{'='*60}")

result = agent.invoke({"messages": [HumanMessage(content=question)]})

# הדפסת שרשרת המחשבה
for msg in result["messages"]:
    role = type(msg).__name__.replace("Message", "")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] 🔍 מחפש: {tc['args'].get('query', '')}")
    elif role == "Tool":
        print(f"[{role}] תוצאות חיפוש התקבלו")
    else:
        print(f"[{role}]\n{msg.content}")
    print("-" * 40)